In [ ]:
#------------------------------------------------ Begin_Librairie ----------------------------------------
import datetime
import os
import re
from time import sleep
from urllib.parse import urljoin
import pandas as pd
import pdfplumber
import requests
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.support.ui import WebDriverWait


In [ ]:
#------------------------------------------------ Begin_ fileName ----------------------------------------
regulatorName = 'WS CBSAM'

print(f"Running {regulatorName} Web Scraping Tool v.1.1")
now = datetime.datetime.now()
filename = '{} SQL Ready {}.xlsx'.format(regulatorName, str(now).replace(':', '.')[:-7])

scriptfolder = f"C:\\Users\\wuj1\\OneDrive - Moody's\\Desktop\\Regulator\\{regulatorName}"

#scriptfolder=os.path.dirname(os.path.abspath(__file__)) ## to decomment for the production environment
os.chdir(scriptfolder)

tempfolder = os.path.join(scriptfolder, 'tempfolder')
tempfolder_abs = os.path.abspath(tempfolder)
if not tempfolder_abs or tempfolder_abs in [os.path.abspath(os.sep), os.path.abspath(scriptfolder)]:
    raise RuntimeError(f'Refusing to clean unsafe tempfolder path: {tempfolder_abs}')

if os.path.exists(tempfolder_abs):
    for rem in os.listdir(tempfolder_abs):
        os.remove(os.path.join(tempfolder_abs, rem))
else:
    os.mkdir(tempfolder_abs)


Running WS CDSAM Web Scraping Tool v.1.1


In [12]:
#------------------------------------------------ Begin_Variable ----------------------------------------

sqldict = {'bvdid': [], 'priority': [], 'ListLabel': [], 'Typology': [], 'EntryType': [], 'Name': [], 'InternalID_1': [], 'InternalID_1_type': [], 'InternalID_2': [], 
          'InternalID_2_type': [], 'InternalID_3': [], 'InternalID_3_type': [], 'CoType': [], 'License_Type': [], 'Address_1': [], 'Address_2': [], 'City': [], 
          'Zip': [], 'Cntry': [], 'Phone': [], 'Fax': [], 'Website': [], 'Email': [], 'RegulationType': [], 'RegulationTypeCode': [], 'RegulationDate': [], 'CancellationDate': [], 
          'RegCtry': [], 'RegCode' : [], 'ListCode': [], 'ListLanguage': [], 'ListValidityDate': [], 'ListName': [], 'ListProcessDate': [], 'LEI Code': [], 'BIC SWIFT Code': [], 'Name - Mother Company': [],
          'Address_1 - Mother company': [], 'Address_2 -  Mother company': [], 'City - Mother company': [], 'Zip - Mother company': [], 'Cntry - Mother company': [], 
          'Phone - Mother company': [], 'Check': []}

processdate = now.strftime('%Y-%m-%d')

regdict = {
    regulatorName + ' 1': 'https://cbs.gov.ws/financial-supervision-regulation',
}

Typology = {
    regulatorName + ' 1': 'LICENSED FINANCIAL INSTITUTIONS',
}


In [13]:
#------------------------------------------------ Begin_Fouction ----------------------------------------
def bourange_same_length_array(sqldict):
    maxlen = len(sqldict['ListProcessDate'])
    for key, val in sqldict.items():
        if len(sqldict[key]) != maxlen:
            empty = []
            total_empty = maxlen - len(sqldict[key])
            for i in range(total_empty):
                empty.append('')
            sqldict[key] = sqldict[key] + empty
    return sqldict


def clean_text(value):
    if value is None:
        return ''
    value = re.sub(r'\s+', ' ', str(value)).strip()
    value = value.strip(' -:\u2022\t\n\r')
    return value


def dedupe_key(name):
    return re.sub(r'[^A-Z0-9]+', '', clean_text(name).upper())


In [14]:
#------------------------------------------------ Begin_chromedriver ----------------------------------------
# Chrome is only used when the source page blocks a direct requests call.
chromeOptions = webdriver.ChromeOptions()
chromeOptions.add_argument('--start-maximized')


In [15]:
#------------------------------------------------ Begin_Main ----------------------------------------

session = requests.Session()
session.headers.update({
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0 Safari/537.36',
    'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,application/pdf;q=0.8,*/*;q=0.7',
})

for reg, url in regdict.items():
    list_name = Typology[reg]
    print(f'[INFO] : Working {reg} - {list_name}')

    # 1. Open source page and find the PDF under "List of Licensed Financial Institutions".
    try:
        response = session.get(url, timeout=60)
        response.raise_for_status()
        html = response.text
    except Exception as exc:
        print(f'[WARN] : Direct source page request failed: {exc}')
        print('[INFO] : Falling back to Selenium for source page discovery')
        driver = webdriver.Chrome(options=chromeOptions)
        try:
            driver.get(url)
            WebDriverWait(driver, 30).until(lambda d: d.execute_script('return document.readyState') == 'complete')
            sleep(3)
            driver.execute_script('window.scrollTo(0, document.body.scrollHeight);')
            sleep(2)
            html = driver.page_source
        finally:
            driver.quit()

    soup = BeautifulSoup(html, 'html.parser')
    pdf_candidates = []
    for anchor in soup.find_all('a', href=True):
        href = anchor['href']
        link_text = clean_text(anchor.get_text(' ', strip=True))
        combined_text = f'{link_text} {href}'.lower()
        if '.pdf' not in combined_text:
            continue

        score = 0
        if 'licensed' in combined_text:
            score += 3
        if 'financial' in combined_text:
            score += 3
        if 'institution' in combined_text:
            score += 3
        if 'list' in combined_text:
            score += 1
        pdf_candidates.append((score, urljoin(url, href), link_text))

    if not pdf_candidates:
        raise RuntimeError('No PDF link found on the source page.')

    pdf_candidates.sort(key=lambda item: item[0], reverse=True)
    pdf_url = pdf_candidates[0][1]
    print(f'[INFO] : Selected PDF link: {pdf_url}')

    # 2. Download the PDF into tempfolder.
    pdf_response = session.get(pdf_url, timeout=90)
    pdf_response.raise_for_status()
    pdf_name = os.path.basename(pdf_url.split('?')[0]) or 'licensed_financial_institutions.pdf'
    pdf_path = os.path.join(tempfolder, pdf_name)
    with open(pdf_path, 'wb') as pdf_file:
        pdf_file.write(pdf_response.content)
    print(f'[INFO] : Downloaded PDF to {pdf_path}')

    # 3. Process downloaded PDF files from tempfolder.
    extracted_rows = []
    for downloaded_file in os.listdir(tempfolder):
        file_path = os.path.join(tempfolder, downloaded_file)
        if not downloaded_file.lower().endswith('.pdf'):
            continue

        print(f'[INFO] : Processing downloaded file: {downloaded_file}')
        with pdfplumber.open(file_path) as pdf:
            for page in pdf.pages:
                for table in page.extract_tables() or []:
                    for row in table:
                        cells = [clean_text(cell) for cell in row]
                        if len(cells) < 5:
                            continue

                        try:
                            int(cells[0])
                        except Exception:
                            continue

                        license_value = cells[1]
                        name_value = cells[2]
                        address_value = cells[3]
                        contact_value = cells[4]
                        name_lower = name_value.lower()

                        if not name_value or name_lower in ['name', 'institution', 'institution name', 'name of institution']:
                            continue
                        if any(term in name_lower for term in ['licensed financial institutions', 'central bank', 'list of']):
                            continue

                        email_match = re.search(r'[A-Za-z0-9._%+\-]+@[A-Za-z0-9.\-]+\.[A-Za-z]{2,}', contact_value)
                        email_value = email_match.group(0) if email_match else ''

                        phone_value = re.sub(r'[A-Za-z0-9._%+\-]+@[A-Za-z0-9.\-]+\.[A-Za-z]{2,}', '', contact_value)
                        phone_value = re.sub(r'(?i)email\s*:?', '', phone_value)
                        phone_value = re.sub(r'(?i)(tel|telephone|phone|contact|fax)\s*:?', '', phone_value)
                        phone_value = clean_text(phone_value)
                        if not re.search(r'\d', phone_value):
                            phone_value = ''

                        extracted_rows.append({
                            'Name': name_value,
                            'InternalID_1': license_value,
                            'InternalID_1_type': 'Licence Number',
                            'Address_1': address_value,
                            'Phone': phone_value,
                            'Email': email_value,
                            'SourceFile': downloaded_file,
                        })

    raw_df = pd.DataFrame(extracted_rows)
    if raw_df.empty:
        raise RuntimeError('No entities extracted from downloaded PDF files.')

    raw_df['dedupe_key'] = raw_df['Name'].apply(dedupe_key)
    raw_df = raw_df[raw_df['dedupe_key'].str.len() > 0]
    raw_df = raw_df.drop_duplicates(subset='dedupe_key').reset_index(drop=True)
    print(f'[INFO] : Extracted {len(raw_df)} unique entities from downloaded PDF files')

    # 4. Append clean, deduplicated PDF rows into SQL-ready structure.
    for i, row in raw_df.iterrows():
        sqldict['Name'].append(row['Name'])
        sqldict['InternalID_1'].append(row['InternalID_1'])
        sqldict['InternalID_1_type'].append(row['InternalID_1_type'])
        sqldict['Address_1'].append(row['Address_1'])
        sqldict['Phone'].append(row['Phone'])
        sqldict['Email'].append(row['Email'])
        sqldict['ListProcessDate'].append(processdate)
        sqldict['RegCtry'].append('WS')
        sqldict['RegCode'].append('CDSAM')
        sqldict['ListCode'].append(reg.split(' ')[-1])
        sqldict['RegulationType'].append('Regulated')
        sqldict['ListName'].append(list_name)
        sqldict = bourange_same_length_array(sqldict)

# 5. Delete only downloaded scratch files under tempfolder after processing.
tempfolder_abs = os.path.abspath(tempfolder)
if not tempfolder_abs or tempfolder_abs in [os.path.abspath(os.sep), os.path.abspath(scriptfolder)]:
    raise RuntimeError(f'Refusing to clean unsafe tempfolder path: {tempfolder_abs}')

for rem in os.listdir(tempfolder_abs):
    os.remove(os.path.join(tempfolder_abs, rem))
print('[INFO] : Cleaned tempfolder after processing downloads')


[INFO] : Working WS CDSAM 1 - LICENSED FINANCIAL INSTITUTIONS
[INFO] : Selected PDF link: https://cbs.gov.ws/media/Licensed-Financial-Institutions-2026-Updated.pdf
[INFO] : Downloaded PDF to C:\Users\wuj1\OneDrive - Moody's\Desktop\Regulator\WS CDSAM\tempfolder\Licensed-Financial-Institutions-2026-Updated.pdf
[INFO] : Processing downloaded file: Licensed-Financial-Institutions-2026-Updated.pdf
[INFO] : Extracted 46 unique entities from downloaded PDF files
[INFO] : Cleaned tempfolder after processing downloads


In [16]:
#------------------------------------------------ Save DataFrame to Excel ----------------------------------------
os.chdir(scriptfolder)
df = pd.DataFrame(sqldict)
df.to_excel(filename, 'SQL Ready', index=False)
sleep(3)
print(f"[INFO] : Excel file '{filename}' saved successfully")


C:\Users\wuj1\AppData\Local\Temp\1\ipykernel_58324\4114852132.py:4: FutureWarning: Starting with pandas version 3.0 all arguments of to_excel except for the argument 'excel_writer' will be keyword-only.
  df.to_excel(filename, 'SQL Ready', index=False)


[INFO] : Excel file 'WS CDSAM SQL Ready 2026-04-27 15.25.40.xlsx' saved successfully


In [17]:
raw_df

,Name,InternalID_1,InternalID_1_type,Address_1,Phone,Email,SourceFile,dedupe_key
0,ANZ (Samoa) Limited,CB001,Licence Number,PO Box L1855 Apia,69999 Online enquiry,,Licensed-Financial-Institutions-2026-Updated.pdf,ANZSAMOALIMITED
1,Bank South Pacific (BSP),CB002,Licence Number,PO Box 1860 Apia,66100,BSPSamoa@bsp.com.pg,Licensed-Financial-Institutions-2026-Updated.pdf,BANKSOUTHPACIFICBSP
2,National Bank of Samoa (NBS),CB003,Licence Number,PO Box 3047L APIA,26766,info@nbsamoa.com,Licensed-Financial-Institutions-2026-Updated.pdf,NATIONALBANKOFSAMOANBS
3,Samoa Commercial Bank (SCB),CB004,Licence Number,PO Box L602 Apia,31233,info@scbl.ws,Licensed-Financial-Institutions-2026-Updated.pdf,SAMOACOMMERCIALBANKSCB
4,FEXCO (SAMOA) LIMITED,MTO001,Licence Number,Head Office Federal Pacific Building Saleufi,25623,esamau@fexcopacific.com,Licensed-Financial-Institutions-2026-Updated.pdf,FEXCOSAMOALIMITED
5,SAMOA FINANCE COMPANY LIMITED,MTO002,Licence Number,PO Box 173 Matafele,22316,suitupe@samoaonline.ws,Licensed-Financial-Institutions-2026-Updated.pdf,SAMOAFINANCECOMPANYLIMITED
6,PACIFIC EZY (PacificEzy),MTO003,Licence Number,DBS Building Ground Floor,20149/20150,info@pacificezygroup.com,Licensed-Financial-Institutions-2026-Updated.pdf,PACIFICEZYPACIFICEZY
7,PACIFIC WAY MONEY TRANSFER LIMITED,MTO004,Licence Number,Lauofo Complex Saleufi,28475,pacway@ipasifika.net,Licensed-Financial-Institutions-2026-Updated.pdf,PACIFICWAYMONEYTRANSFERLIMITED
8,IMEX SAMOA MONEY TRANSFER,MTO006,Licence Number,Wesley Arcade Matafele,7667840,imexsamoa2016@gmail.com,Licensed-Financial-Institutions-2026-Updated.pdf,IMEXSAMOAMONEYTRANSFER
9,SAMOA MONEY TRANSFER LIMITED,MTO007,Licence Number,Farmer Joe car park Fugalei,27694,info@samoamoneytransfer.co.nz,Licensed-Financial-Institutions-2026-Updated.pdf,SAMOAMONEYTRANSFERLIMITED


In [18]:
#------------------------------------------------ Data Integrity & Consistency Check ----------------------------------------

print('=' * 80)
print('DATA INTEGRITY & CONSISTENCY VERIFICATION')
print('=' * 80)

print(f'\n1. DATAFRAME SHAPE:')
print(f'   Total rows collected: {len(df)}')
print(f'   Total columns: {len(df.columns)}')

print('\n2. DATA DISTRIBUTION BY LIST:')
if len(df) > 0:
    list_summary = df.groupby('ListCode').agg({
        'Name': 'count',
        'ListName': 'first',
        'RegCtry': 'first',
        'RegCode': 'first'
    }).rename(columns={'Name': 'Count'})
    print(list_summary)
else:
    print('   No rows collected')

print('\n3. REQUIRED VALUES CHECK:')
if len(df) > 0:
    print(f"   RegCtry values: {df['RegCtry'].unique()}")
    print(f"   RegCode values: {df['RegCode'].unique()}")
    print(f"   ListCode values: {df['ListCode'].unique()}")
    print(f"   Duplicate entity names: {df['Name'].duplicated().sum()}")
    print('\n4. SAMPLE DATA:')
    print(df[['Name', 'RegCtry', 'RegCode', 'ListCode', 'ListName']].head(10).to_string(index=False))
else:
    print('   No data available for validation')

print('\n' + '=' * 80)
print('SUMMARY')
print('=' * 80)
print(f'Total unique licensed financial institutions collected: {len(df)}')


DATA INTEGRITY & CONSISTENCY VERIFICATION

1. DATAFRAME SHAPE:
   Total rows collected: 46
   Total columns: 44

2. DATA DISTRIBUTION BY LIST:
          Count                         ListName RegCtry RegCode
ListCode                                                        
1            46  LICENSED FINANCIAL INSTITUTIONS      WS   CDSAM

3. REQUIRED VALUES CHECK:
   RegCtry values: ['WS']
   RegCode values: ['CDSAM']
   ListCode values: ['1']
   Duplicate entity names: 0

4. SAMPLE DATA:
                              Name RegCtry RegCode ListCode                        ListName
               ANZ (Samoa) Limited      WS   CDSAM        1 LICENSED FINANCIAL INSTITUTIONS
          Bank South Pacific (BSP)      WS   CDSAM        1 LICENSED FINANCIAL INSTITUTIONS
      National Bank of Samoa (NBS)      WS   CDSAM        1 LICENSED FINANCIAL INSTITUTIONS
       Samoa Commercial Bank (SCB)      WS   CDSAM        1 LICENSED FINANCIAL INSTITUTIONS
             FEXCO (SAMOA) LIMITED      WS   CDS